# Baseline Tool-Call Evaluation

This notebook runs the seed baseline evaluation against the base model using the same `litellm.completion` path as the exploratory notebook.

It records:
- the prompt
- expected tool and args
- raw model response
- binary scores
- total reward from `0` to `4`

It also saves JSONL and CSV copies under `outputs/`.

In [1]:
from pathlib import Path
import csv
import json
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toolcall_rl.evaluation.baseline import run_baseline

MODEL = "ollama_chat/smollm:1.7b"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL

08:37:28 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
08:37:28 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


'ollama_chat/smollm:1.7b'

## Run Baseline

This calls Ollama through LiteLLM for each seed eval case.

In [2]:
results = run_baseline(model=MODEL)
len(results)

10

## Build Table

In [3]:
def flatten_result(index, result):
    case = result["case"]
    score = result["score"]
    return {
        "case_id": index,
        "prompt": case["prompt"],
        "expected_tool": case["expected_tool"],
        "expected_args": json.dumps(case["expected_args"], sort_keys=True),
        "response": result["response"],
        "valid_json": score["valid_json"],
        "json_only": score["json_only"],
        "tool_match": score["tool_match"],
        "args_match": score["args_match"],
        "total_reward": score["total_reward"],
        "parsed": json.dumps(score["parsed"], sort_keys=True),
    }

table_rows = [flatten_result(index, result) for index, result in enumerate(results, start=1)]
table_rows[:1]

[{'case_id': 1,
  'prompt': 'What is 24 * 17?',
  'expected_tool': 'calculator',
  'expected_args': '{"expression": "24 * 17"}',
  'response': 'The answer to this question is: 24 * 17 = 398',
  'valid_json': 0,
  'json_only': 0,
  'tool_match': 0,
  'args_match': 0,
  'total_reward': 0,
  'parsed': 'null'}]

## Summary

In [4]:
total_cases = len(table_rows)
passed_cases = sum(row["total_reward"] == 4 for row in table_rows)
total_reward = sum(row["total_reward"] for row in table_rows)
max_reward = total_cases * 4

{
    "model": MODEL,
    "cases": total_cases,
    "passed": passed_cases,
    "total_reward": total_reward,
    "max_reward": max_reward,
}

{'model': 'ollama_chat/smollm:1.7b',
 'cases': 10,
 'passed': 0,
 'total_reward': 0,
 'max_reward': 40}

## Results Table

In [5]:
import pandas as pd

df = pd.DataFrame(table_rows)
df[
    [
        "case_id",
        "expected_tool",
        "valid_json",
        "json_only",
        "tool_match",
        "args_match",
        "total_reward",
        "prompt",
        "response",
    ]
]

,case_id,expected_tool,valid_json,json_only,tool_match,args_match,total_reward,prompt,response
0,1,calculator,0,0,0,0,0,What is 24 * 17?,The answer to this question is: 24 * 17 = 398
1,2,calculator,0,0,0,0,0,Calculate (892 + 431) / 3 using the calculator.,The calculator can be used to calculate the re...
2,3,google_search,0,0,0,0,0,Search Google for recent news about open sourc...,Here are some recent news articles about open ...
3,4,google_search,0,0,0,0,0,Use Google to find information about the Googl...,The Google AdWords Knowledge Base (ADK) is a c...
4,5,unit_converter,0,0,0,0,0,Convert 10 kilometers to miles.,"To convert 10 kilometers to miles, we need to ..."
5,6,unit_converter,0,0,0,0,0,How many pounds are in 7 kilograms?,There are 14.592358 kilograms in 7 kilograms.
6,7,text_stats,0,0,0,0,0,"Count the words and sentences in: ""Hello world...",Here's how you can count the words and sentenc...
7,8,text_stats,0,0,0,0,0,"How many characters are in this text: ""small m...",The number of characters in the given text is ...
8,9,string_formatter,0,0,0,0,0,"Make this title case: ""learning tool calls""",Learning Tool Calls
9,10,string_formatter,0,0,0,0,0,"Reverse this text: ""stressed""","The reversed text of ""stressed"" is ""esst"""


## Save Results

In [6]:
jsonl_path = OUTPUT_DIR / "baseline_eval_results.jsonl"
csv_path = OUTPUT_DIR / "baseline_eval_results.csv"

with jsonl_path.open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=True) + "\n")

with csv_path.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)

{
    "jsonl": str(jsonl_path),
    "csv": str(csv_path),
}

{'jsonl': '/home/shubeeksh/projects/toolcall-rl/outputs/baseline_eval_results.jsonl',
 'csv': '/home/shubeeksh/projects/toolcall-rl/outputs/baseline_eval_results.csv'}

### Observed failure modes:

1. Ignores JSON-only instruction
2. Answers directly instead of tool calling
3. Explains tools instead of invoking them
4. Outputs incorrect schema
5. Outputs natural language around tool usage
6. Hallucinates weather data
7. Sometimes partially follows format
